# Day 2 — Explore the Dataset

**Goal:** Understand the data before we touch any AI code.  
Good engineers always explore first, build second.

---

## 📥 Step 0 — Download the Dataset First

Run this in the Shell **before** running any cells here:

```bash
cd fake-news-detector && python data/download_data.py
```

When it says `✅ Download complete!` — come back and run the cells below.

---

**Dataset info:**
- ~7,000 English news articles
- Columns: `title`, `text`, `label` (REAL or FAKE)
- Source: George McIntire / lutzhamel (public domain, GitHub)


In [ ]:
# ── Cell 1: Import the tools we need ──────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Make charts look nicer
sns.set_theme(style='whitegrid')

print('✅ All imports successful!')

In [ ]:
# ── Cell 2: Load the dataset ───────────────────────────────────────────────────
# This dataset has 3 columns: title, text, label (REAL or FAKE)

df = pd.read_csv('../data/raw/news.csv')

# The label column says 'REAL' or 'FAKE' — convert to numbers for the model
# 0 = REAL,  1 = FAKE
df['label_num'] = df['label'].map({'REAL': 0, 'FAKE': 1})

print(f'✅ Loaded {len(df):,} articles')
print(f'   REAL : {(df["label"]=="REAL").sum():,}')
print(f'   FAKE : {(df["label"]=="FAKE").sum():,}')

In [ ]:
# ── Cell 3: Peek at the first 5 rows ──────────────────────────────────────────
# .head() shows the first 5 rows — like glancing at the top of a spreadsheet

df.head()

In [ ]:
# ── Cell 4: Check shape and column names ──────────────────────────────────────

print('Shape (rows × columns):', df.shape)
print('Columns :', df.columns.tolist())
print()
print(df.dtypes)

In [ ]:
# ── Cell 5: Check for missing values ──────────────────────────────────────────
# Missing values (NaN) cause errors during training — we need to catch them now

print('Missing values per column:')
print(df.isnull().sum())
print()
print('Total rows with ANY missing value:', df.isnull().any(axis=1).sum())

In [ ]:
# ── Cell 6: Read a real FAKE article vs a real REAL article ───────────────────
# This helps us understand with our own eyes what the AI needs to learn

print('=' * 60)
print('🔴 FAKE ARTICLE SAMPLE')
print('=' * 60)
fake_sample = df[df['label'] == 'FAKE'].iloc[0]
print('Title:', fake_sample['title'])
print()
print('Text (first 400 chars):')
print(str(fake_sample['text'])[:400])

print()
print('=' * 60)
print('🟢 REAL ARTICLE SAMPLE')
print('=' * 60)
real_sample = df[df['label'] == 'REAL'].iloc[0]
print('Title:', real_sample['title'])
print()
print('Text (first 400 chars):')
print(str(real_sample['text'])[:400])

In [ ]:
# ── Cell 7: Bar chart — REAL vs FAKE count ────────────────────────────────────
# A balanced dataset trains a better model

counts = df['label'].value_counts()

plt.figure(figsize=(6, 4))
bars = plt.bar(counts.index, counts.values,
               color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.5)

# Add count labels on top of each bar
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{val:,}', ha='center', fontsize=12, fontweight='bold')

plt.title('Article Count: REAL vs FAKE', fontsize=14)
plt.ylabel('Number of Articles')
plt.tight_layout()
plt.savefig('../data/processed/chart_class_balance.png', dpi=100)
plt.show()

print('Balance ratio:', round(counts.min() / counts.max(), 2),
      '  (1.0 = perfectly balanced)')

In [ ]:
# ── Cell 8: Article text length comparison ────────────────────────────────────
# Are fake articles longer or shorter than real ones?
# Patterns in length can be features the AI picks up

df['text_length'] = df['text'].fillna('').apply(len)

plt.figure(figsize=(9, 4))
df[df['label'] == 'REAL']['text_length'].plot(
    kind='hist', bins=50, alpha=0.6, color='#2ecc71', label='REAL')
df[df['label'] == 'FAKE']['text_length'].plot(
    kind='hist', bins=50, alpha=0.6, color='#e74c3c', label='FAKE')
plt.title('Article Length: REAL vs FAKE', fontsize=14)
plt.xlabel('Number of Characters')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('../data/processed/chart_text_length.png', dpi=100)
plt.show()

print('Average length — REAL:', int(df[df['label']=='REAL']['text_length'].mean()))
print('Average length — FAKE:', int(df[df['label']=='FAKE']['text_length'].mean()))

In [ ]:
# ── Cell 9: Word Cloud — FAKE news ────────────────────────────────────────────
# Bigger word = appears more often in fake articles

fake_text = ' '.join(df[df['label'] == 'FAKE']['text'].fillna('').values)

wc = WordCloud(width=800, height=350, background_color='white',
               colormap='Reds', max_words=120).generate(fake_text)

plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in FAKE Articles', fontsize=16)
plt.tight_layout()
plt.savefig('../data/processed/wordcloud_fake.png', dpi=100)
plt.show()
print('💡 Notice: what kinds of words dominate fake news?')

In [ ]:
# ── Cell 10: Word Cloud — REAL news ───────────────────────────────────────────

real_text = ' '.join(df[df['label'] == 'REAL']['text'].fillna('').values)

wc = WordCloud(width=800, height=350, background_color='white',
               colormap='Greens', max_words=120).generate(real_text)

plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in REAL Articles', fontsize=16)
plt.tight_layout()
plt.savefig('../data/processed/wordcloud_real.png', dpi=100)
plt.show()
print('💡 Notice: how are the words different from fake news words?')

In [ ]:
# ── Cell 11: Summary ──────────────────────────────────────────────────────────

counts = df['label'].value_counts()

print('=' * 50)
print('📊 DATASET SUMMARY')
print('=' * 50)
print(f'Total articles       : {len(df):,}')
print(f'REAL articles        : {counts.get("REAL", 0):,}')
print(f'FAKE articles        : {counts.get("FAKE", 0):,}')
print(f'Missing text values  : {df["text"].isnull().sum()}')
print(f'Columns              : {df.columns.tolist()}')
print()
print('Charts saved to data/processed/')
print()
print('✅ Day 2 complete!')
print('   Next → Day 3: write data/preprocess.py to clean this data')